# 07 - Demo de prospeccion real con LangGraph a Excel

Este notebook crea prospectos para una categoria especifica y una cantidad especifica. Usa LangGraph, valida campos completos, evita duplicados contra `data/agent_memory.sqlite3` y guarda el resultado en Excel dentro de `data/`.

## Flujo de la demo

1. Configurar categoria y cantidad.
2. Construir input de Apify sin ejecutar todavia.
3. Mostrar grafo LangGraph legible.
4. Ejecutar prospeccion real solo si `EJECUTAR_PROSPECCION_REAL=True`.
5. Exportar Excel y registrar memoria local.

In [ ]:
from pathlib import Path
import os
import sys
import json

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Carga .env sin imprimir secretos.
env_path = ROOT / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8-sig').splitlines():
        if line.strip() and not line.strip().startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip())

import pandas as pd
from IPython.display import Markdown, display

from src.agents.prospecting_graph import compiled_prospecting_graph
from src.db import repository


## 1. Parametros de categoria y cantidad

Cambia estos valores segun la demo. Ejemplos de categoria: `Hogar`, `Moda`, `Vehiculos`, `Belleza`, `Tecnologia`.

In [ ]:
CATEGORIA_OBJETIVO = 'Hogar'
CANTIDAD_PROSPECTOS = 3

SEGMENTOS = {
    'Hogar': {
        'industries': ['Retail', 'Furniture', 'Consumer Goods'],
        'keywords': ['hogar', 'muebles', 'decoracion', 'home'],
        'titles': ['Founder', 'CEO', 'Director', 'Head of Ecommerce', 'Ecommerce Manager'],
    },
    'Moda': {
        'industries': ['Apparel & Fashion', 'Retail', 'Consumer Goods'],
        'keywords': ['moda', 'fashion', 'ropa', 'calzado'],
        'titles': ['Founder', 'CEO', 'Commercial Director', 'Head of Ecommerce', 'Ecommerce Manager'],
    },
    'Vehiculos': {
        'industries': ['Automotive', 'Retail'],
        'keywords': ['autopartes', 'vehiculos', 'motos', 'repuestos'],
        'titles': ['Founder', 'CEO', 'Commercial Director', 'Head of Ecommerce'],
    },
    'Belleza': {
        'industries': ['Cosmetics', 'Retail', 'Consumer Goods'],
        'keywords': ['belleza', 'cosmeticos', 'skincare', 'maquillaje'],
        'titles': ['Founder', 'CEO', 'Brand Manager', 'Head of Ecommerce'],
    },
    'Tecnologia': {
        'industries': ['Consumer Electronics', 'Retail', 'Technology'],
        'keywords': ['tecnologia', 'computadores', 'celulares', 'electronics'],
        'titles': ['Founder', 'CEO', 'Commercial Director', 'Head of Ecommerce'],
    },
}

if CATEGORIA_OBJETIVO not in SEGMENTOS:
    raise ValueError(f'Categoria no configurada: {CATEGORIA_OBJETIVO}. Opciones: {list(SEGMENTOS)}')

segmento = SEGMENTOS[CATEGORIA_OBJETIVO]
print(f'Categoria: {CATEGORIA_OBJETIVO}')
print(f'Cantidad objetivo: {CANTIDAD_PROSPECTOS}')

## 2. Input de Apify antes de gastar la llamada

El input se muestra primero para revision. No se envia nada por Slack, email ni WhatsApp.

In [ ]:
run_input = {
    'totalResults': int(CANTIDAD_PROSPECTOS),
    'hasEmail': True,
    'emailStatusIncludes': ['verified'],
    'hasPhone': True,
    'personLocationCountryIncludes': ['Colombia'],
    'companyLocationCountryIncludes': ['Colombia'],
    'companyIndustryIncludes': segmento['industries'],
    'personTitleIncludes': segmento['titles'],
    'roleMatchMode': 'any',
    'companyKeywordIncludes': segmento['keywords'],
    'companyKeywordMode': 'broad',
    'resetProgress': False,
    'dontSaveProgress': True,
    'countOnly': False,
}

print(json.dumps(run_input, ensure_ascii=False, indent=2))

## 3. Grafo LangGraph de prospeccion

El grafo busca en Apify, valida campos, deduplica contra SQLite, redacta perfil/borrador con Groq y exporta Excel.

In [ ]:
def mermaid_legible(raw: str) -> str:
    theme = """%%{init: {'theme': 'base', 'themeVariables': {'primaryColor': '#FFFFFF', 'primaryTextColor': '#111827', 'primaryBorderColor': '#2563EB', 'lineColor': '#334155', 'secondaryColor': '#EEF2FF', 'tertiaryColor': '#F8FAFC', 'fontFamily': 'Arial', 'fontSize': '18px'}}}%%"""
    styled = raw.replace('classDef default fill:#f2f0ff,line-height:1.2', 'classDef default fill:#FFFFFF,stroke:#2563EB,color:#111827,line-height:1.35,font-size:18px')
    styled += "\nclassDef first fill:#DBEAFE,stroke:#1D4ED8,color:#111827,font-weight:bold,font-size:18px;"
    styled += "\nclassDef last fill:#DCFCE7,stroke:#15803D,color:#111827,font-weight:bold,font-size:18px;"
    return theme + '\n' + styled

mermaid_prospeccion = mermaid_legible(compiled_prospecting_graph.get_graph().draw_mermaid())
display(Markdown('```mermaid\n' + mermaid_prospeccion + '\n```'))
print(mermaid_prospeccion)

## 4. Ejecutar prospeccion real

Activa `EJECUTAR_PROSPECCION_REAL=True` solo cuando el input de Apify este correcto. El resultado se guarda en `data/test2_prospectos_apify.xlsx` y los consultados quedan en la base local para evitar duplicados.

In [ ]:
EJECUTAR_PROSPECCION_REAL = False

faltantes = [key for key in ['APIFY_API_TOKEN', 'GROQ_API_KEY'] if not os.environ.get(key)]
if faltantes:
    print('Faltan variables para ejecucion real:', faltantes)

if EJECUTAR_PROSPECCION_REAL:
    state = compiled_prospecting_graph.invoke({
        'run_input': run_input,
        'max_results': int(CANTIDAD_PROSPECTOS),
        'log': [],
    })
    print('\n'.join(state.get('log', [])))
    print('Excel:', state.get('output_path'))
    display(pd.DataFrame(state.get('enriched_rows', [])))
else:
    state = None
    print('Modo seguro: no se llamo Apify ni Groq. Cambia EJECUTAR_PROSPECCION_REAL=True para ejecutar.')

## 5. Memoria y deduplicacion

La tabla `prospect_consultations` evita volver a consultar los mismos contactos. La tabla `agent_interactions` deja trazabilidad de la corrida.

In [ ]:
display(pd.DataFrame(repository.list_agent_interactions(limit=20)))

if state:
    for row in state.get('enriched_rows', []):
        print(row.get('contact_email'), 'duplicado_en_db=', repository.prospect_exists(contact_email=row.get('contact_email')))